# Workstream C: Neuron-Level XAI and Information Flow



## 1. Mount Drive and set up

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, glob, re, json, csv, math, shutil
from collections import Counter, defaultdict
import numpy as np

MYDRIVE = "/content/drive/MyDrive"
ROOT = None
for c in sorted(glob.glob(os.path.join(MYDRIVE, "**", "WISER Results"), recursive=True)):
    if os.path.isdir(c): ROOT = c; break
if ROOT is None:
    ROOT = os.path.join(MYDRIVE, "WISER Results"); os.makedirs(ROOT, exist_ok=True)
OUT = os.path.join(ROOT, "Phase 3", "WSC")
os.makedirs(OUT, exist_ok=True)
print("WISER Results :", ROOT)
print("output folder :", OUT)

N_SLICES = 26
T_SLICES = np.round(np.linspace(0.0, 1.0, N_SLICES), 4)
print("time slices (%d):" % N_SLICES, T_SLICES[:4], "...", T_SLICES[-2:])

ACCOUNT, REBUILT, NS, SRC_NB = [], {}, None, None

WISER Results : /content/drive/MyDrive/WISER Results
output folder : /content/drive/MyDrive/WISER Results/Phase 3/WSC
time slices (26): [0.   0.04 0.08 0.12] ... [0.96 1.  ]


## 2. Import the model definitions from your training notebooks


In [3]:
# ---------------------------------------------------------------------------
# Import the model CLASSES only. Two safeguards, because a first attempt at this
# accidentally started a training sweep:
#   1. Any cell that CALLS the training machinery is never executed.
#   2. Execution stops as soon as the classes we need exist.
# extract_activations is NOT imported; it is implemented locally in cell 5, so
# no cell after the model definitions ever has to run.
# ---------------------------------------------------------------------------
def is_driver(src):
    """True if the cell CALLS training machinery rather than defining it."""
    for name in ("run_all", "run_one", "run_sweep", "train_schedule", "fit", "main"):
        if (name + "(") in src and ("def " + name) not in src:
            return True
    return False

def load_definitions(nb_path, needed):
    nb = json.load(open(nb_path)); g = {"__name__": "__main__"}; n = 0
    for c in nb["cells"]:
        if c["cell_type"] != "code": continue
        src = "".join(c["source"])
        if is_driver(src):
            print("   [stop] driver cell reached, not executed"); break
        if "drive.mount" in src: continue
        clean = "\n".join(l for l in src.splitlines()
                          if not l.strip().startswith(("!", "%", "get_ipython")))
        try:
            exec(compile(clean, "<cell%d>" % n, "exec"), g)
        except Exception as e:
            print("   [warn] cell %d: %s" % (n, type(e).__name__))
        n += 1
        if all(k in g for k in needed):
            print("   [ok] definitions complete after %d cells, stopping early" % n); break
    return g, n

cands = [p for p in glob.glob(os.path.join(MYDRIVE, "**", "*.ipynb"), recursive=True)
         if any(k in os.path.basename(p) for k in
                ("GAAF", "PHASE2_FINAL", "Heat", "heat", "EXTENDED", "hardnu",
                 "N1_", "N3_", "N4_"))]
print("candidate training notebooks: %d" % len(cands))
for p in cands[:12]: print("   ", p.replace(MYDRIVE, "").lstrip("/"))

# Pass 1 wants all four classes so GAAF runs can be rebuilt too. Pass 2 settles
# for the three essential ones, and GAAF runs are then reported as unavailable
# rather than crashing the analysis.
ESSENTIAL = ["QAPINN", "ClassicalTwin", "TWIN_CONFIG"]
PREFERRED = ESSENTIAL + ["GAAFPINN"]
order = sorted(cands, key=lambda q: (("GAAF" not in q), len(q)))

for label, needed in (("pass 1 (all four classes)", PREFERRED),
                      ("pass 2 (essential classes only)", ESSENTIAL)):
    print("\n--- %s ---" % label)
    for p in order:
        print("trying %s ..." % os.path.basename(p))
        try:
            g, n = load_definitions(p, needed)
        except Exception as e:
            print("   failed: %s" % type(e).__name__); continue
        print("   found:", [k for k in PREFERRED if k in g])
        if all(k in g for k in needed):
            NS, SRC_NB = g, p
            print("   [OK] using %s" % os.path.basename(p)); break
    if NS is not None: break

if NS is None:
    print("\n[STOP] Could not import the model classes. Do not run the cells below;")
    print("       send me the candidate list above.")
else:
    have = [k for k in ("QAPINN", "ClassicalTwin", "GAAFPINN", "GAAFTanh") if k in NS]
    print("\navailable:", have)
    if "GAAFPINN" not in NS:
        print("[WARNING] GAAFPINN is not available from this notebook, so GAAF-PINN runs")
        print("          cannot be rebuilt. They will be reported as unavailable and")
        print("          excluded, and every GAAF figure will be omitted rather than")
        print("          silently left out. QAPINN and twin analysis is unaffected.")
    print("NOTE: no training was executed. Only class and helper definitions were run.")

candidate training notebooks: 10
    WISER Results/Phase 2/Djabon Phase 2/Tiers12_q38_more_iter/QAPINN_Djabon_v5_EXTENDED_SWEEP.ipynb
    WISER Results/Phase 2/Djabon Phase 2/Tiers12_multi_seed/QAPINN_Djabon_PHASE2_FINAL.ipynb
    WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/GAAF-PINN/QAPINN_Djabon_GAAF.ipynb
    WISER Results/Phase 2 Hard Burger/GAAF-PINN/N4_GAAF_hardnu_q3to8.ipynb
    WISER Results/Phase 2 Hard Burger/MERGE RESULTS/N5_MERGE_hardnu.ipynb
    WISER Results/Phase 2 Hard Burger/c-PINN/N3_TWIN_hardnu_q3to8.ipynb
    WISER Results/Phase 2 Hard Burger/QAPINN N6/N2q6_QAPINN_hardnu_q6.ipynb
    WISER Results/Phase 2 Hard Burger/QAPINN N8/N2q8_QAPINN_hardnu_q8.ipynb
    WISER Results/Phase 2 Hard Burger/QAPINN N7/N2q7_QAPINN_hardnu_q7.ipynb
    WISER Results/Phase 2 Hard Burger/QAPINN N35/N1_QAPINN_hardnu_q345.ipynb

--- pass 1 (all four classes) ---
trying N4_GAAF_hardnu_q3to8.ipynb ...
[OK] Phase 3 directory: quapinns/phase3_hardnu_gaaf
[ok] torch
[..] insta

## 3. Inventory the checkpoints, with full accounting

In [4]:
# Multi-seed runs.
RE_MS = re.compile(r"MS_(?P<hard>hard_)?(?P<pde>burgers|heat)_n(?P<n>\d+)_"
                   r"(?P<model>qapinn|twin|gaaf)_s(?P<seed>\d+)(?P<retry>_r\d+)?")
# The V5 extended sweep is the ONLY QAPINN data at q=6,7,8 on Burgers. An earlier
# version of this notebook accepted MS_ tags only, which silently discarded it and
# left the wide-circuit QAPINN panels empty. These runs are single-seed (the global
# SEED=1234), which every figure and table must state.
RE_V5 = re.compile(r"V5_(?P<pde>burgers|heat)_q(?P<n>\d+)")
RE_T1 = re.compile(r"T1_(?P<pde>burgers|heat)_q(?P<n>\d+)")
RE_CT = re.compile(r"CT_(?P<pde>burgers|heat)_n(?P<n>\d+)")

def parse_tag(tag):
    m = RE_MS.search(tag)
    if m:
        d = m.groupdict()
        return dict(pde=d["pde"], n_feat=int(d["n"]), model=d["model"], seed=int(d["seed"]),
                    nu=("0.01/pi" if d["hard"] else "0.05"), retry=bool(d["retry"]),
                    family="MS", single_seed=False)
    m = RE_V5.search(tag)
    if m:
        return dict(pde=m.group("pde"), n_feat=int(m.group("n")), model="qapinn", seed=1234,
                    nu="0.05", retry=False, family="V5", single_seed=True)
    m = RE_T1.search(tag)
    if m:
        return dict(pde=m.group("pde"), n_feat=int(m.group("n")), model="qapinn", seed=1234,
                    nu="0.05", retry=False, family="T1", single_seed=True)
    m = RE_CT.search(tag)
    if m:
        return dict(pde=m.group("pde"), n_feat=int(m.group("n")), model="twin", seed=1234,
                    nu="0.05", retry=False, family="CT", single_seed=True)
    return None

CKPTS = glob.glob(os.path.join(MYDRIVE, "**", "checkpoints", "*.pt"), recursive=True)
print("checkpoint files found: %d" % len(CKPTS))

seen, TARGETS = {}, []
for p in sorted(CKPTS):
    tag = os.path.basename(p)[:-3]; rel = p.replace(MYDRIVE, "").lstrip("/")
    meta = parse_tag(tag)
    if meta is None:
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason="tag matched no known naming convention")); continue
    if tag in seen:
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason="duplicate tag, already taken from %s" % seen[tag])); continue
    seen[tag] = rel; TARGETS.append((tag, p, meta))
    ACCOUNT.append(dict(tag=tag, path=rel, decision="selected", reason=""))

sel = [a for a in ACCOUNT if a["decision"] == "selected"]
exc = [a for a in ACCOUNT if a["decision"] == "excluded"]
print("  selected : %d\n  excluded : %d" % (len(sel), len(exc)))
for r, c in Counter(a["reason"] for a in exc).most_common():
    print("     %4d  %s" % (c, r))
assert len(sel) + len(exc) == len(CKPTS), "ACCOUNTING FAILURE: files lost silently"
print("[OK] accounting closes: %d + %d = %d" % (len(sel), len(exc), len(CKPTS)))

cov = defaultdict(list)
for tag, _, m in TARGETS:
    lbl = ("%dr" % m["seed"]) if m["retry"] else str(m["seed"])
    if m.get("single_seed"): lbl += "*"
    cov[(m["nu"], m["pde"], m["model"], m["n_feat"])].append(lbl)
print("\ncoverage  (* = single-seed run, from the V5/T1/CT families):")
for k in sorted(cov, key=lambda z: (z[0], z[1], z[2], z[3])):
    print("  nu=%-8s %-8s %-7s n=%d: %s" % (k[0], k[1], k[2], k[3], sorted(cov[k])))
print("\nby family:", dict(Counter(m["family"] for _, _, m in TARGETS)))

checkpoint files found: 301
  selected : 214
  excluded : 87
       54  tag matched no known naming convention
        2  duplicate tag, already taken from WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/GAAF-PINN/qapinn_runs_gaaf/checkpoints/MS_heat_n4_twin_s1234.pt
        2  duplicate tag, already taken from WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/GAAF-PINN/qapinn_runs_gaaf/checkpoints/MS_heat_n4_twin_s2025.pt
        2  duplicate tag, already taken from WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/GAAF-PINN/qapinn_runs_gaaf/checkpoints/MS_heat_n4_twin_s777.pt
        1  duplicate tag, already taken from WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/GAAF-PINN/qapinn_runs_gaaf/checkpoints/MS_heat_n3_twin_s1234.pt
        1  duplicate tag, already taken from WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/GAAF-PINN/qapinn_runs_gaaf/checkpoints/MS_heat_n3_twin_s2025.pt
        1  duplicate tag, already t

## 4. Rebuild each model and extract activations at every slice

Forward passes only. Any run that fails to rebuild is recorded by name and excluded.

In [ ]:
import torch
import torch.nn as nn

QAPINN = NS["QAPINN"]; ClassicalTwin = NS["ClassicalTwin"]; GAAFPINN = NS.get("GAAFPINN")
device = NS.get("device", torch.device("cpu")); DTYPE = NS.get("DTYPE", torch.float32)

def linear_cka(X, Y):
    X = np.asarray(X, float); Y = np.asarray(Y, float)
    if X.ndim == 1: X = X.reshape(-1, 1)
    if Y.ndim == 1: Y = Y.reshape(-1, 1)
    X = X - X.mean(0, keepdims=True); Y = Y - Y.mean(0, keepdims=True)
    num = np.linalg.norm(Y.T @ X, "fro") ** 2
    den = np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")
    return float(num / den) if den > 0 else 0.0

# --- self-test of the CKA implementation against its defining properties ----
_rng = np.random.default_rng(0)
_X = _rng.normal(size=(64, 8)); _Q, _ = np.linalg.qr(_rng.normal(size=(8, 8)))
assert abs(linear_cka(_X, _X) - 1) < 1e-9,        "CKA(X,X) must be 1"
assert abs(linear_cka(_X, _X @ _Q) - 1) < 1e-9,   "CKA must be orthogonally invariant"
assert abs(linear_cka(_X, 7.3 * _X) - 1) < 1e-9,  "CKA must be scale invariant"
assert linear_cka(_X, np.ones((64, 3))) == 0.0,   "degenerate input must give 0"
print("[ok] linear CKA verified: identity, orthogonal invariance, scale invariance, degeneracy")

# --- activation extraction, implemented here rather than imported ------------
@torch.no_grad()
def extract_acts(model, t_slice, n_grid=256):
    """Activations of the first layer and of every hidden nonlinearity, plus the output.
    Layer names match those used by the training pipeline: L0_quantum or L0_front,
    then L1..L5 from the five hidden nonlinearities of the head."""
    xs = np.linspace(-1, 1, n_grid)
    xin = torch.tensor(xs.reshape(-1, 1), dtype=DTYPE, device=device)
    tin = torch.full_like(xin, float(t_slice))
    store, hooks = {}, []
    first, fname = getattr(model, "qlayer", None), "L0_quantum"
    if first is None:
        first, fname = getattr(model, "front", None), "L0_front"
    if first is not None:
        hooks.append(first.register_forward_hook(
            lambda m, i, o, k=fname: store.__setitem__(k, o.detach().cpu().numpy())))
    k = 0
    for mod in model.head:
        if isinstance(mod, nn.Tanh) or mod.__class__.__name__ == "GAAFTanh":
            k += 1
            hooks.append(mod.register_forward_hook(
                lambda m, i, o, kk=k: store.__setitem__("L%d" % kk,
                                                        o.detach().cpu().numpy())))
    out = model(xin, tin).cpu().numpy()
    for h in hooks: h.remove()
    return store, xs, out

AVAILABLE_KINDS = {"qapinn", "twin"} | ({"gaaf"} if GAAFPINN is not None else set())
print("model kinds that can be rebuilt:", sorted(AVAILABLE_KINDS))

def build(kind, n):
    """Return a fresh model, or None if its class was not importable."""
    if kind == "qapinn":
        return QAPINN(n_qubits=n, n_layers=6, entanglement="all",
                      measurement="expval", head_width=20, head_depth=5)
    if kind == "twin":
        return ClassicalTwin(n_feat=n, head_width=20, head_depth=5)
    if kind == "gaaf":
        return None if GAAFPINN is None else GAAFPINN(n_feat=n, head_width=20, head_depth=5)
    return None

# --- probe the hook layout on one model of each available kind --------------
for _kind in sorted(AVAILABLE_KINDS):
    _t = next(((tg, pa, mm) for tg, pa, mm in TARGETS if mm["model"] == _kind), None)
    if _t is None:
        print("[probe] no %s run in the corpus" % _kind); continue
    _tag, _path, _m = _t
    try:
        _ck = torch.load(_path, map_location="cpu", weights_only=False)
        _mdl = build(_kind, _m["n_feat"]); _mdl.load_state_dict(_ck["model_state"]); _mdl.eval()
        _a, _x, _o = extract_acts(_mdl, 0.5)
        print("[probe] %-38s layers %s out %s"
              % (_tag, {k: tuple(v.shape) for k, v in _a.items()}, tuple(_o.shape)))
        assert len(_a) >= 2, "hooks captured too few layers for %s" % _kind
    except Exception as e:
        print("[probe] %s FAILED: %s: %s" % (_kind, type(e).__name__, str(e)[:90]))

ROWS, NEURON, failures = [], [], []
for i, (tag, path, meta) in enumerate(TARGETS):
    if meta["model"] not in AVAILABLE_KINDS:
        failures.append((tag, "model class %s was not importable" % meta["model"])); continue
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        model = build(meta["model"], meta["n_feat"])
        if model is None:
            failures.append((tag, "build returned None for %s" % meta["model"])); continue
        model.load_state_dict(ck["model_state"]); model.eval().to(device)
    except Exception as e:
        failures.append((tag, "rebuild: %s: %s" % (type(e).__name__, str(e)[:80]))); continue
    try:
        for t in T_SLICES:
            acts, xs, out = extract_acts(model, float(t))
            names = list(acts)
            row = dict(tag=tag, t=float(t)); row.update(meta)
            for nm in names:
                row["cka_%s_out" % nm] = linear_cka(acts[nm], out)
            row["cka_first_last"] = linear_cka(acts[names[0]], acts[names[-1]])
            ROWS.append(row)
            for nm in names:
                A = acts[nm]
                for j in range(A.shape[1]):
                    v = A[:, j]
                    nr = dict(tag=tag, t=float(t), layer=nm, neuron=j,
                              mean=float(v.mean()), std=float(v.std()),
                              amin=float(v.min()), amax=float(v.max()),
                              sat=float(np.mean(np.abs(v) > 0.95)),
                              cka_out=linear_cka(v, out))
                    nr.update(meta); NEURON.append(nr)
    except Exception as e:
        failures.append((tag, "activations: %s: %s" % (type(e).__name__, str(e)[:80]))); continue
    REBUILT[tag] = True
    if (i + 1) % 20 == 0:
        print("  %d/%d runs processed" % (i + 1, len(TARGETS)))

print("\nrebuilt successfully : %d" % len(REBUILT))
print("failures             : %d" % len(failures))
for t, w in failures[:15]: print("   [fail] %s: %s" % (t, w))
if len(failures) > 15: print("   ... and %d more" % (len(failures) - 15))
print("CKA rows    : %d\nneuron rows : %d" % (len(ROWS), len(NEURON)))

[ok] linear CKA verified: identity, orthogonal invariance, scale invariance, degeneracy
model kinds that can be rebuilt: ['gaaf', 'qapinn', 'twin']
[probe] MS_hard_burgers_n3_gaaf_s1234          layers {'L0_front': (256, 3), 'L1': (256, 20), 'L2': (256, 20), 'L3': (256, 20), 'L4': (256, 20), 'L5': (256, 20)} out (256, 1)
[probe] MS_hard_burgers_n3_qapinn_s1234        layers {'L0_quantum': (256, 3), 'L1': (256, 20), 'L2': (256, 20), 'L3': (256, 20), 'L4': (256, 20), 'L5': (256, 20)} out (256, 1)
[probe] MS_hard_burgers_n3_twin_s1234          layers {'L0_front': (256, 3), 'L1': (256, 20), 'L2': (256, 20), 'L3': (256, 20), 'L4': (256, 20), 'L5': (256, 20)} out (256, 1)
  20/214 runs processed
  40/214 runs processed
  60/214 runs processed
  80/214 runs processed
  100/214 runs processed
  120/214 runs processed
  140/214 runs processed
  160/214 runs processed
  180/214 runs processed
  200/214 runs processed


## 5. Save the numerical export

In [ ]:
def dump(rows, name):
    if not rows:
        print("[skip]", name, "(no rows)"); return
    p = os.path.join(OUT, name); keys = sorted({k for r in rows for k in r})
    with open(p, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
    print("[saved] %s (%d rows)" % (p, len(rows)))

dump(ROWS, "wsc_cka_by_slice.csv")
dump(NEURON, "wsc_neuron_stats.csv")
dump(ACCOUNT, "wsc_checkpoint_accounting.csv")

## 6. Figures

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9.5, "axes.titlesize": 10.5,
    "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.fontsize": 8.2, "legend.frameon": True, "legend.framealpha": 0.92,
    "legend.edgecolor": "0.8", "axes.grid": True, "grid.alpha": 0.20,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "savefig.dpi": 300})
C_Q, C_T, C_G = "#14285a", "#c0392b", "#2e8b57"
STYLE = {"qapinn": (C_Q, "QAPINN", "o"), "twin": (C_T, "Classical twin", "s"),
         "gaaf": (C_G, "GAAF-PINN", "^")}
def savefig(fig, name):
    p = os.path.join(OUT, name + ".png")
    fig.savefig(p, bbox_inches="tight"); plt.close(fig); print("[fig]", p)

NU = "0.05"
R = [r for r in ROWS if r["nu"] == NU]

def first_key(r):
    for k in r:
        if k.startswith("cka_") and k.endswith("_out") and \
           ("L0" in k or "quantum" in k or "front" in k):
            return k
    return None

# ---- C1: the decisive figure -------------------------------------------
for pde in sorted({r["pde"] for r in R}):
    ns = sorted({r["n_feat"] for r in R if r["pde"] == pde})
    if not ns: continue
    ncol = 3; nrow = math.ceil(len(ns) / ncol)
    fig, axs = plt.subplots(nrow, ncol, figsize=(4.0*ncol, 3.1*nrow),
                            sharey=True, sharex=True, squeeze=False)
    for idx, n in enumerate(ns):
        ax = axs[idx // ncol][idx % ncol]
        for kind in ["qapinn", "twin", "gaaf"]:
            sub = [r for r in R if r["pde"] == pde and r["n_feat"] == n and r["model"] == kind]
            if not sub: continue
            col, lab, _ = STYLE[kind]
            byseed = defaultdict(list)
            for r in sub:
                k = first_key(r)
                if k: byseed[r["seed"]].append((r["t"], r[k]))
            for s, pts in byseed.items():
                pts.sort(); ax.plot([a for a, _ in pts], [b for _, b in pts],
                                    color=col, alpha=0.55, lw=1.0)
            ax.plot([], [], color=col, label="%s (%d)" % (lab, len(byseed)))
        ax.set_title("$n=%d$" % n, loc="left"); ax.set_ylim(-0.02, 1.02)
    axs[0][0].legend(loc="best", fontsize=7.5)
    for j in range(ncol): axs[nrow-1][j].set_xlabel("$t$")
    for i2 in range(nrow): axs[i2][0].set_ylabel("CKA(first layer, output)")
    for idx in range(len(ns), nrow*ncol): axs[idx // ncol][idx % ncol].axis("off")
    fig.suptitle("WS-C: how much of the first layer survives into the prediction, "
                 "%s at $\\nu=0.05$ (one line per seed)" % pde, y=1.005)
    plt.tight_layout(); savefig(fig, "C1_cka_first_layer_to_output_%s" % pde)

# ---- C2: CKA against width ---------------------------------------------
if R:
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    for kind in ["qapinn", "twin", "gaaf"]:
        sub = [r for r in R if r["model"] == kind]
        if not sub: continue
        col, lab, mk = STYLE[kind]
        ns = sorted({r["n_feat"] for r in sub}); mu, sd = [], []
        for n in ns:
            vals = [r[first_key(r)] for r in sub if r["n_feat"] == n and first_key(r)]
            mu.append(np.mean(vals)); sd.append(np.std(vals))
        ax.errorbar(ns, mu, yerr=sd, color=col, marker=mk, ms=5, lw=1.6, capsize=3, label=lab)
    ax.set_xlabel("qubit count / feature width $n$")
    ax.set_ylabel("CKA(first layer, output)")
    ax.set_title(r"WS-C: does the first layer matter more as it widens? ($\nu=0.05$)",
                 loc="left")
    ax.legend(loc="best"); savefig(fig, "C2_cka_vs_width")

# ---- C3: CKA by depth ---------------------------------------------------
if R:
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    for kind in ["qapinn", "twin", "gaaf"]:
        sub = [r for r in R if r["model"] == kind]
        if not sub: continue
        col, lab, mk = STYLE[kind]
        keys = sorted({k for r in sub for k in r
                       if k.startswith("cka_") and k.endswith("_out")},
                      key=lambda s: (len(s), s))
        mu = [np.mean([r[k] for r in sub if k in r]) for k in keys]
        ax.plot(range(len(keys)), mu, color=col, marker=mk, ms=5, lw=1.6, label=lab)
        ax.set_xticks(range(len(keys)))
        ax.set_xticklabels([k[4:-4] for k in keys], rotation=45, fontsize=8)
    ax.set_ylabel("CKA(layer, output)")
    ax.set_title("WS-C: where the output representation is formed", loc="left")
    ax.legend(loc="best"); savefig(fig, "C3_cka_by_depth")

# ---- C4: Figure-3 reproduction -----------------------------------------
def fig3(pde, n, seed, t=0.5):
    trip = []
    for kind in ["qapinn", "twin", "gaaf"]:
        for tg, _, m in TARGETS:
            if (m["nu"] == NU and m["pde"] == pde and m["n_feat"] == n and
                    m["model"] == kind and m["seed"] == seed and tg in REBUILT):
                trip.append((kind, tg)); break
    if not trip:
        print("[skip] Figure-3 %s n=%d s=%d" % (pde, n, seed)); return
    tags = [tg for _, tg in trip]
    ts = min(T_SLICES, key=lambda z: abs(z - t))
    sub = [r for r in NEURON if r["t"] == float(ts) and r["layer"] in ("L2", "L3")
           and r["tag"] in tags]
    if not sub:
        print("[skip] Figure-3 %s n=%d: no neuron rows" % (pde, n)); return
    fig, axs = plt.subplots(2, len(trip), figsize=(4.2*len(trip), 6.2), squeeze=False)
    for c_, (kind, tag) in enumerate(trip):
        col, lab, _ = STYLE[kind]
        for r_, layer in enumerate(["L2", "L3"]):
            ax = axs[r_][c_]
            vals = sorted([x for x in sub if x["tag"] == tag and x["layer"] == layer],
                          key=lambda z: z["neuron"])
            ax.bar([v["neuron"] for v in vals], [v["std"] for v in vals],
                   color=col, alpha=0.85)
            ax.set_title("%s, layer %s" % (lab, layer), loc="left", fontsize=9.5)
            ax.set_xlabel("neuron index"); ax.set_ylabel("activation std. dev.")
    fig.suptitle("WS-C: per-neuron activation spread, %s $n=%d$, seed %d, $t=%.2f$\n"
                 "(the brief's Figure-3 comparison, with and without the quantum layer)"
                 % (pde, n, seed, ts), y=1.01)
    plt.tight_layout(); savefig(fig, "C4_figure3_%s_n%d_s%d" % (pde, n, seed))

for pde in sorted({r["pde"] for r in R}):
    ns = sorted({r["n_feat"] for r in R if r["pde"] == pde})
    if ns: fig3(pde, ns[len(ns)//2], 1234)

# ---- C5: saturation by layer -------------------------------------------
NN = [r for r in NEURON if r["nu"] == NU]
if NN:
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    for kind in ["qapinn", "twin", "gaaf"]:
        sub = [r for r in NN if r["model"] == kind]
        if not sub: continue
        col, lab, mk = STYLE[kind]
        layers = sorted({r["layer"] for r in sub}, key=lambda s: (len(s), s))
        mu = [np.mean([r["sat"] for r in sub if r["layer"] == L]) for L in layers]
        ax.plot(range(len(layers)), mu, color=col, marker=mk, ms=5, lw=1.6, label=lab)
        ax.set_xticks(range(len(layers))); ax.set_xticklabels(layers, rotation=45, fontsize=8)
    ax.set_ylabel(r"fraction of inputs with $|a|>0.95$")
    ax.set_title("WS-C: neuron saturation by layer", loc="left")
    ax.legend(loc="best"); savefig(fig, "C5_saturation_by_layer")

## 7. Manifest and archive

In [ ]:
man = dict(
    note="wsc-neuron-xai (Track A, Djabon)",
    source_notebook=os.path.basename(SRC_NB) if SRC_NB else None,
    n_slices=N_SLICES, t_slices=[float(t) for t in T_SLICES],
    checkpoints_found=len(CKPTS), selected=len(sel), excluded=len(exc),
    rebuilt=len(REBUILT), n_failures=len(failures),
    failures=[{"tag": a, "reason": b} for a, b in failures],
    n_cka_rows=len(ROWS), n_neuron_rows=len(NEURON),
    coverage={"%s|%s|%s|n%d" % (k[0], k[1], k[2], k[3]): sorted(v) for k, v in cov.items()},
    caveats=[
        "Activations were RE-EXTRACTED from checkpoints at 26 slices; the stored *_acts.npz "
        "files (single slice t=0.5) were not used.",
        "Model classes were imported from a training notebook, not re-typed.",
        "nu=0.05 and nu=0.01/pi are catalogued separately and never merged.",
        "Runs that failed to rebuild are listed by name and excluded from every figure.",
        "CKA is linear CKA, invariant to orthogonal transform and isotropic scaling.",
        "No value is interpolated or filled; missing configurations are reported as missing.",
    ])
json.dump(man, open(os.path.join(OUT, "wsc_manifest.json"), "w"), indent=2)
print(json.dumps({k: man[k] for k in ("source_notebook", "n_slices", "checkpoints_found",
                                      "selected", "excluded", "rebuilt", "n_failures",
                                      "n_cka_rows", "n_neuron_rows")}, indent=2))

NB = "WSC_NeuronXAI_Colab.ipynb"
cands2 = [q for q in glob.glob(os.path.join(MYDRIVE, "**", "*.ipynb"), recursive=True)
          if os.path.basename(q).startswith("WSC_NeuronXAI_Colab")]
if cands2:
    src = max(cands2, key=os.path.getmtime); dst = os.path.join(OUT, NB)
    if os.path.abspath(src) != os.path.abspath(dst): shutil.copy2(src, dst)
    print("\n[saved] notebook archived to", dst)
else:
    print("\n[note] notebook not found in Drive; use File > Save a copy in Drive and re-run.")

print("\nFinal contents of", OUT)
for f in sorted(os.listdir(OUT)):
    print("   %-44s %12d bytes" % (f, os.path.getsize(os.path.join(OUT, f))))